# Reviewer Split Experiments

This notebook generates the additional corpus organizations used to address the ENIAC 2026 reviewer comments.

The original Puntuguese corpus is always read from `data/original/` and is never overwritten.

The generated strategies are:

1. `pair_controlled`: both members of each micro-edited pair remain in the same split;
2. `random_instance`: instances are redistributed independently of pair membership, while preserving the exact original split sizes and class balance;
3. `max_cross_split`: the number of pairs crossing splits is maximized while preserving the exact original split sizes and class balance. In this construction, every validation/test instance has its paired counterpart in training.

Five predefined split seeds are used: **13, 21, 40, 42, 73**.

The models and their hyperparameters are not changed by this notebook.


In [6]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

SPLIT_SEEDS = [13, 21, 40, 42, 73]

SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

STRATEGIES = [
    "pair_controlled",
    "random_instance",
    "max_cross_split",
]

print("Split seeds:", SPLIT_SEEDS)
print("Strategies:", STRATEGIES)


Split seeds: [13, 21, 40, 42, 73]
Strategies: ['pair_controlled', 'random_instance', 'max_cross_split']


In [7]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "original").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected to find a directory containing data/original/."
    )


PROJECT_ROOT = find_project_root()

# IMPORTANT: this is the directory that was missing in the previous version.
ORIGINAL_DIR = PROJECT_ROOT / "data" / "original"

REVIEW_SPLITS_DIR = PROJECT_ROOT / "data" / "review_splits"

ORIGINAL_FILES = {
    "train": ORIGINAL_DIR / "train.jsonl",
    "validation": ORIGINAL_DIR / "validation.jsonl",
    "test": ORIGINAL_DIR / "test.jsonl",
}

for strategy in STRATEGIES:
    for seed in SPLIT_SEEDS:
        (
            REVIEW_SPLITS_DIR
            / strategy
            / f"seed_{seed}"
        ).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Original corpus directory:", ORIGINAL_DIR)
print("Review splits directory:", REVIEW_SPLITS_DIR)

for split_name, path in ORIGINAL_FILES.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing original {split_name} file: {path}"
        )
    print(f"{split_name:10s}: {path}")


Project root: /home/avelar/pun-detection-split-analysis
Original corpus directory: /home/avelar/pun-detection-split-analysis/data/original
Review splits directory: /home/avelar/pun-detection-split-analysis/data/review_splits
train     : /home/avelar/pun-detection-split-analysis/data/original/train.jsonl
validation: /home/avelar/pun-detection-split-analysis/data/original/validation.jsonl
test      : /home/avelar/pun-detection-split-analysis/data/original/test.jsonl


In [8]:
def load_jsonl(path, split_name):
    rows = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            item = json.loads(line)
            item["original_split"] = split_name
            rows.append(item)

    return rows


all_rows = []

for split_name in SPLIT_NAMES:
    all_rows.extend(
        load_jsonl(
            path=ORIGINAL_FILES[split_name],
            split_name=split_name,
        )
    )

df = pd.DataFrame(all_rows)

# Columns that belong to the original JSONL schema.
SOURCE_COLUMNS = [
    column
    for column in df.columns
    if column != "original_split"
]

print("Total examples:", len(df))
print("Original JSONL columns:", SOURCE_COLUMNS)

display(df.head())


Total examples: 5700
Original JSONL columns: ['id', 'text', 'label', 'tokens', 'labels']


,id,text,label,tokens,labels,original_split
0,5.46.H,Por que o carteiro foi à feira? Porque tinha u...,1,"[Por, que, o, carteiro, foi, à, feira, ?, Porq...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0]",train
1,5.792.H,Por que a mulher esotérica não conseguia engra...,1,"[Por, que, a, mulher, esotérica, não, consegui...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",train
2,5.1811.H,Qual é o animal que está sempre cansado? Dorme...,1,"[Qual, é, o, animal, que, está, sempre, cansad...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",train
3,5.733.H,Qual o sambista passou a dar presente pra todo...,1,"[Qual, o, sambista, passou, a, dar, presente, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",train
4,4.652.N,Um homem matou uma ovelha e agora foi preso . ...,0,"[Um, homem, matou, uma, ovelha, e, agora, foi,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",train


In [9]:
REQUIRED_COLUMNS = {
    "id",
    "text",
    "label",
    "original_split",
}

missing_columns = REQUIRED_COLUMNS - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if df["id"].duplicated().any():
    duplicated_ids = (
        df.loc[df["id"].duplicated(keep=False), "id"]
        .astype(str)
        .tolist()
    )
    raise ValueError(
        "Duplicated example IDs were found. "
        f"Examples: {duplicated_ids[:10]}"
    )

df["label"] = df["label"].astype(int)

if set(df["label"].unique()) != {0, 1}:
    raise ValueError(
        f"Unexpected labels: {sorted(df['label'].unique())}"
    )

if len(df) != 5700:
    raise ValueError(
        f"Expected 5700 examples, found {len(df)}."
    )

print("Basic corpus validation passed.")


Basic corpus validation passed.


In [10]:
PAIR_PATTERN = re.compile(
    r"^(?P<pair_id>.+)\.(?P<suffix>[HN])$"
)


def parse_example_id(example_id):
    match = PAIR_PATTERN.match(str(example_id))

    if match is None:
        raise ValueError(
            f"Unexpected example ID format: {example_id}"
        )

    return (
        match.group("pair_id"),
        match.group("suffix"),
    )


parsed_ids = df["id"].apply(parse_example_id)

df["pair_id"] = parsed_ids.apply(lambda value: value[0])
df["pair_suffix"] = parsed_ids.apply(lambda value: value[1])

display(
    df[
        [
            "id",
            "pair_id",
            "pair_suffix",
            "label",
            "original_split",
            "text",
        ]
    ].head()
)


,id,pair_id,pair_suffix,label,original_split,text
0,5.46.H,5.46,H,1,train,Por que o carteiro foi à feira? Porque tinha u...
1,5.792.H,5.792,H,1,train,Por que a mulher esotérica não conseguia engra...
2,5.1811.H,5.1811,H,1,train,Qual é o animal que está sempre cansado? Dorme...
3,5.733.H,5.733,H,1,train,Qual o sambista passou a dar presente pra todo...
4,4.652.N,4.652,N,0,train,Um homem matou uma ovelha e agora foi preso . ...


In [11]:
print("Total examples:", len(df))
print("Unique pairs:", df["pair_id"].nunique())

if df["pair_id"].nunique() != 2850:
    raise ValueError(
        f"Expected 2850 pairs, found {df['pair_id'].nunique()}."
    )

pair_sizes = df.groupby("pair_id").size()

if not (pair_sizes == 2).all():
    invalid_pairs = pair_sizes[pair_sizes != 2]
    raise ValueError(
        "Every IDB must contain exactly two examples.\n"
        f"{invalid_pairs.head()}"
    )

suffix_sets = (
    df.groupby("pair_id")["pair_suffix"]
    .agg(lambda values: frozenset(values))
)

if not (
    suffix_sets == frozenset({"H", "N"})
).all():
    raise ValueError(
        "Every IDB must contain exactly one .H and one .N instance."
    )

label_sets = (
    df.groupby("pair_id")["label"]
    .agg(lambda values: frozenset(values))
)

if not (
    label_sets == frozenset({0, 1})
).all():
    raise ValueError(
        "Every IDB must contain exactly labels 0 and 1."
    )

expected_label_by_suffix = {
    "H": 1,
    "N": 0,
}

suffix_label_ok = df.apply(
    lambda row: (
        expected_label_by_suffix[row["pair_suffix"]]
        == row["label"]
    ),
    axis=1,
)

if not suffix_label_ok.all():
    raise ValueError(
        "At least one .H/.N suffix is inconsistent with its label."
    )

print("Pair structure validation passed:")
print("- 2,850 IDBs")
print("- exactly 2 examples per IDB")
print("- exactly one .H and one .N per IDB")
print("- .H -> label 1")
print("- .N -> label 0")


Total examples: 5700
Unique pairs: 2850
Pair structure validation passed:
- 2,850 IDBs
- exactly 2 examples per IDB
- exactly one .H and one .N per IDB
- .H -> label 1
- .N -> label 0


In [12]:
original_split_counts = (
    df["original_split"]
    .value_counts()
    .reindex(SPLIT_NAMES)
)

print("Original split sizes:")
display(original_split_counts)

if original_split_counts.to_dict() != EXPECTED_SPLIT_COUNTS:
    raise ValueError(
        "Original split sizes do not match the expected sizes."
    )

original_class_distribution = (
    pd.crosstab(
        df["original_split"],
        df["label"],
    )
    .reindex(SPLIT_NAMES)
    .reindex(columns=[0, 1], fill_value=0)
)

print("Original split/class distribution:")
display(original_class_distribution)

for split_name, expected in EXPECTED_CLASS_COUNTS.items():
    observed = (
        df.loc[
            df["original_split"] == split_name,
            "label",
        ]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    if observed != expected:
        raise ValueError(
            f"Unexpected class counts in original {split_name}: "
            f"{observed}; expected {expected}."
        )

print("Original split sizes and class balance validated.")


Original split sizes:


original_split
train         3990
validation     570
test          1140
Name: count, dtype: int64

Original split/class distribution:


label,0,1
original_split,,
train,1995,1995
validation,285,285
test,570,570


Original split sizes and class balance validated.


In [13]:
def make_pair_split_table(dataframe, split_column):
    pair_table = dataframe.pivot(
        index="pair_id",
        columns="pair_suffix",
        values=split_column,
    )

    return pair_table[["H", "N"]]


def make_pair_matrix(dataframe, split_column):
    pair_table = make_pair_split_table(
        dataframe=dataframe,
        split_column=split_column,
    )

    matrix = (
        pd.crosstab(
            pair_table["H"],
            pair_table["N"],
        )
        .reindex(
            index=SPLIT_NAMES,
            columns=SPLIT_NAMES,
            fill_value=0,
        )
    )

    return matrix


def count_cross_split_pairs(dataframe, split_column):
    pair_table = make_pair_split_table(
        dataframe=dataframe,
        split_column=split_column,
    )

    cross_split_mask = (
        pair_table["H"]
        != pair_table["N"]
    )

    cross_split_pairs = int(
        cross_split_mask.sum()
    )

    cross_split_rate = (
        cross_split_pairs
        / len(pair_table)
    )

    return cross_split_pairs, cross_split_rate


original_pair_matrix = make_pair_matrix(
    dataframe=df,
    split_column="original_split",
)

original_cross_split_pairs, original_cross_split_rate = (
    count_cross_split_pairs(
        dataframe=df,
        split_column="original_split",
    )
)

print("Original H x N split matrix:")
display(original_pair_matrix)

print(
    "Original cross-split pairs:",
    original_cross_split_pairs,
)
print(
    "Original cross-split rate:",
    f"{original_cross_split_rate:.4%}",
)

if original_cross_split_pairs != 1306:
    raise ValueError(
        "Expected 1306 cross-split pairs in the original corpus, "
        f"found {original_cross_split_pairs}."
    )

print("Original pair organization validated.")


Original H x N split matrix:


N,train,validation,test
H,,,
train,1404,206,385
validation,196,22,67
test,395,57,118


Original cross-split pairs: 1306
Original cross-split rate: 45.8246%
Original pair organization validated.


In [14]:
def make_json_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)

    if isinstance(obj, np.ndarray):
        return [
            make_json_serializable(value)
            for value in obj.tolist()
        ]

    if isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(value)
            for value in obj
        ]

    if isinstance(obj, dict):
        return {
            str(key): make_json_serializable(value)
            for key, value in obj.items()
        }

    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass

    return obj


def save_jsonl(dataframe, output_path):
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_output_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    with temp_output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for _, row in dataframe[SOURCE_COLUMNS].iterrows():
            item = {
                key: make_json_serializable(value)
                for key, value in row.to_dict().items()
            }

            file.write(
                json.dumps(
                    item,
                    ensure_ascii=False,
                )
                + "\n"
            )

    temp_output_path.replace(output_path)


def count_jsonl_lines(path):
    with path.open("r", encoding="utf-8") as file:
        return sum(
            1
            for line in file
            if line.strip()
        )


In [15]:
def validate_generated_split(
    dataframe,
    strategy,
    seed,
):
    if len(dataframe) != 5700:
        raise ValueError(
            f"{strategy}/{seed}: expected 5700 examples, "
            f"found {len(dataframe)}."
        )

    if dataframe["id"].nunique() != 5700:
        raise ValueError(
            f"{strategy}/{seed}: IDs are not unique."
        )

    if set(dataframe["id"]) != set(df["id"]):
        raise ValueError(
            f"{strategy}/{seed}: generated corpus does not contain "
            "exactly the original example IDs."
        )

    if dataframe["pair_id"].nunique() != 2850:
        raise ValueError(
            f"{strategy}/{seed}: expected 2850 pairs."
        )

    observed_split_counts = (
        dataframe["review_split"]
        .value_counts()
        .reindex(SPLIT_NAMES)
        .to_dict()
    )

    if observed_split_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(
            f"{strategy}/{seed}: unexpected split sizes: "
            f"{observed_split_counts}"
        )

    for split_name, expected in EXPECTED_CLASS_COUNTS.items():
        observed = (
            dataframe.loc[
                dataframe["review_split"] == split_name,
                "label",
            ]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed != expected:
            raise ValueError(
                f"{strategy}/{seed}: unexpected class counts in "
                f"{split_name}: {observed}; expected {expected}."
            )

    cross_split_pairs, cross_split_rate = (
        count_cross_split_pairs(
            dataframe=dataframe,
            split_column="review_split",
        )
    )

    pair_matrix = make_pair_matrix(
        dataframe=dataframe,
        split_column="review_split",
    )

    if strategy == "pair_controlled":
        if cross_split_pairs != 0:
            raise ValueError(
                f"{strategy}/{seed}: expected 0 cross-split pairs, "
                f"found {cross_split_pairs}."
            )

    if strategy == "max_cross_split":
        if cross_split_pairs != 1710:
            raise ValueError(
                f"{strategy}/{seed}: expected 1710 cross-split pairs, "
                f"found {cross_split_pairs}."
            )

        # In this specific maximum-cross construction, every validation/test
        # instance must have its paired counterpart in training.
        pair_table = make_pair_split_table(
            dataframe=dataframe,
            split_column="review_split",
        )

        for pair_id, row in pair_table.iterrows():
            if row["H"] != row["N"]:
                if "train" not in {row["H"], row["N"]}:
                    raise ValueError(
                        f"{strategy}/{seed}: cross-split pair {pair_id} "
                        "does not include training."
                    )

    return {
        "strategy": strategy,
        "seed": int(seed),
        "total_examples": int(len(dataframe)),
        "total_pairs": int(dataframe["pair_id"].nunique()),
        "train_examples": int(
            (dataframe["review_split"] == "train").sum()
        ),
        "validation_examples": int(
            (dataframe["review_split"] == "validation").sum()
        ),
        "test_examples": int(
            (dataframe["review_split"] == "test").sum()
        ),
        "cross_split_pairs": int(cross_split_pairs),
        "cross_split_rate": float(cross_split_rate),
        "pair_matrix": pair_matrix,
    }


In [16]:
def save_generated_run(
    dataframe,
    strategy,
    seed,
    validation_summary,
):
    seed_dir = (
        REVIEW_SPLITS_DIR
        / strategy
        / f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # 1) JSONL split files
    for split_name in SPLIT_NAMES:
        split_df = (
            dataframe.loc[
                dataframe["review_split"] == split_name
            ]
            .copy()
        )

        output_path = (
            seed_dir
            / f"{split_name}.jsonl"
        )

        save_jsonl(
            dataframe=split_df,
            output_path=output_path,
        )

    # 2) Pair matrix
    pair_matrix_path = (
        seed_dir
        / "pair_matrix.csv"
    )

    validation_summary["pair_matrix"].to_csv(
        pair_matrix_path,
        encoding="utf-8",
    )

    # 3) Human-readable inspection file
    inspection_columns = [
        "id",
        "pair_id",
        "pair_suffix",
        "label",
        "original_split",
        "review_split",
        "text",
    ]

    inspection_path = (
        seed_dir
        / "inspection.csv"
    )

    dataframe[
        inspection_columns
    ].to_csv(
        inspection_path,
        index=False,
        encoding="utf-8",
    )

    # 4) Metadata
    metadata = {
        key: value
        for key, value in validation_summary.items()
        if key != "pair_matrix"
    }

    metadata["class_distribution"] = {
        split_name: {
            str(label): int(count)
            for label, count in (
                dataframe.loc[
                    dataframe["review_split"] == split_name,
                    "label",
                ]
                .value_counts()
                .sort_index()
                .to_dict()
                .items()
            )
        }
        for split_name in SPLIT_NAMES
    }

    metadata_path = (
        seed_dir
        / "metadata.json"
    )

    with metadata_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    # 5) Verify saved JSONL sizes
    saved_counts = {
        split_name: count_jsonl_lines(
            seed_dir / f"{split_name}.jsonl"
        )
        for split_name in SPLIT_NAMES
    }

    if saved_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(
            f"{strategy}/{seed}: saved JSONL sizes are incorrect: "
            f"{saved_counts}"
        )

    return seed_dir


## Strategy 1 — Pair-controlled split

The pair ID (IDB) is the atomic unit of redistribution. Each pair is assigned entirely to one split.

Because every pair contains exactly two instances (one `.H` and one `.N`) and the target split sizes are all even, the exact original split sizes and the exact 50/50 class balance can be preserved:

- 1,995 pairs → training → 3,990 examples;
- 285 pairs → validation → 570 examples;
- 570 pairs → test → 1,140 examples.


In [17]:
def generate_pair_controlled(
    dataframe,
    seed,
):
    rng = np.random.default_rng(seed)

    pair_ids = np.array(
        sorted(dataframe["pair_id"].unique()),
        dtype=object,
    )

    rng.shuffle(pair_ids)

    train_pair_count = (
        EXPECTED_SPLIT_COUNTS["train"] // 2
    )
    validation_pair_count = (
        EXPECTED_SPLIT_COUNTS["validation"] // 2
    )
    test_pair_count = (
        EXPECTED_SPLIT_COUNTS["test"] // 2
    )

    if (
        train_pair_count
        + validation_pair_count
        + test_pair_count
        != len(pair_ids)
    ):
        raise ValueError(
            "Pair counts derived from target split sizes "
            "do not cover all pairs."
        )

    train_end = train_pair_count
    validation_end = (
        train_end
        + validation_pair_count
    )

    train_pairs = set(
        pair_ids[:train_end]
    )
    validation_pairs = set(
        pair_ids[
            train_end:validation_end
        ]
    )
    test_pairs = set(
        pair_ids[validation_end:]
    )

    result = dataframe.copy()

    def assign_split(pair_id):
        if pair_id in train_pairs:
            return "train"

        if pair_id in validation_pairs:
            return "validation"

        if pair_id in test_pairs:
            return "test"

        raise RuntimeError(
            f"Pair ID was not assigned: {pair_id}"
        )

    result["review_split"] = (
        result["pair_id"]
        .apply(assign_split)
    )

    return result


## Strategy 2 — Random instance-level split

The `.H` and `.N` instances are redistributed independently of pair membership.

To preserve the exact original class balance, positive and negative examples are shuffled separately and then assigned according to the original per-class split counts.


In [18]:
def generate_random_instance(
    dataframe,
    seed,
):
    rng = np.random.default_rng(seed)

    split_by_id = {}

    for label in [0, 1]:
        label_ids = np.array(
            sorted(
                dataframe.loc[
                    dataframe["label"] == label,
                    "id",
                ].tolist()
            ),
            dtype=object,
        )

        rng.shuffle(label_ids)

        train_count = (
            EXPECTED_CLASS_COUNTS["train"][label]
        )
        validation_count = (
            EXPECTED_CLASS_COUNTS["validation"][label]
        )
        test_count = (
            EXPECTED_CLASS_COUNTS["test"][label]
        )

        if (
            train_count
            + validation_count
            + test_count
            != len(label_ids)
        ):
            raise ValueError(
                f"Class {label}: target counts do not cover all examples."
            )

        train_end = train_count
        validation_end = (
            train_end
            + validation_count
        )

        for example_id in label_ids[:train_end]:
            split_by_id[example_id] = "train"

        for example_id in label_ids[
            train_end:validation_end
        ]:
            split_by_id[example_id] = "validation"

        for example_id in label_ids[
            validation_end:
        ]:
            split_by_id[example_id] = "test"

    if len(split_by_id) != len(dataframe):
        raise RuntimeError(
            "Random instance assignment did not cover all examples."
        )

    result = dataframe.copy()

    result["review_split"] = (
        result["id"]
        .map(split_by_id)
    )

    if result["review_split"].isna().any():
        raise RuntimeError(
            "At least one example was not assigned in random_instance."
        )

    return result


## Strategy 3 — Maximally cross-split

With 2,850 pairs and 3,990 training instances, not all pairs can be forced to cross splits: training alone contains more instances than there are pairs.

The maximum is obtained as follows:

- 1,140 pairs remain complete in training;
- the remaining 1,710 pairs are split between training and evaluation;
- 855 `.H` instances are outside training and their `.N` counterparts remain in training;
- 855 `.N` instances are outside training and their `.H` counterparts remain in training;
- validation receives 285 `.H` + 285 `.N`;
- test receives 570 `.H` + 570 `.N`.

Therefore exactly **1,710 / 2,850 = 60%** of pairs cross splits, while every split keeps the original size and 50/50 class balance.


In [19]:
def generate_max_cross_split(
    dataframe,
    seed,
):
    rng = np.random.default_rng(seed)

    pair_ids = np.array(
        sorted(dataframe["pair_id"].unique()),
        dtype=object,
    )

    rng.shuffle(pair_ids)

    # Number of H/N examples that must be outside training.
    h_outside_train_count = (
        EXPECTED_CLASS_COUNTS["validation"][1]
        + EXPECTED_CLASS_COUNTS["test"][1]
    )

    n_outside_train_count = (
        EXPECTED_CLASS_COUNTS["validation"][0]
        + EXPECTED_CLASS_COUNTS["test"][0]
    )

    train_complete_pair_count = (
        len(pair_ids)
        - h_outside_train_count
        - n_outside_train_count
    )

    if train_complete_pair_count != 1140:
        raise ValueError(
            "Expected 1140 complete training pairs in the "
            "max-cross construction."
        )

    if h_outside_train_count != 855:
        raise ValueError(
            "Expected 855 H instances outside training."
        )

    if n_outside_train_count != 855:
        raise ValueError(
            "Expected 855 N instances outside training."
        )

    complete_train_end = (
        train_complete_pair_count
    )

    h_outside_end = (
        complete_train_end
        + h_outside_train_count
    )

    complete_train_pairs = set(
        pair_ids[:complete_train_end]
    )

    h_outside_pairs = list(
        pair_ids[
            complete_train_end:h_outside_end
        ]
    )

    n_outside_pairs = list(
        pair_ids[h_outside_end:]
    )

    # The global pair shuffle already randomized membership; shuffle these
    # subsets again so validation/test membership is independently randomized.
    rng.shuffle(h_outside_pairs)
    rng.shuffle(n_outside_pairs)

    h_validation_count = (
        EXPECTED_CLASS_COUNTS["validation"][1]
    )
    n_validation_count = (
        EXPECTED_CLASS_COUNTS["validation"][0]
    )

    h_validation_pairs = set(
        h_outside_pairs[
            :h_validation_count
        ]
    )
    h_test_pairs = set(
        h_outside_pairs[
            h_validation_count:
        ]
    )

    n_validation_pairs = set(
        n_outside_pairs[
            :n_validation_count
        ]
    )
    n_test_pairs = set(
        n_outside_pairs[
            n_validation_count:
        ]
    )

    split_by_key = {}

    # Complete pairs in training.
    for pair_id in complete_train_pairs:
        split_by_key[
            (pair_id, "H")
        ] = "train"
        split_by_key[
            (pair_id, "N")
        ] = "train"

    # H outside training; N counterpart in training.
    for pair_id in h_validation_pairs:
        split_by_key[
            (pair_id, "H")
        ] = "validation"
        split_by_key[
            (pair_id, "N")
        ] = "train"

    for pair_id in h_test_pairs:
        split_by_key[
            (pair_id, "H")
        ] = "test"
        split_by_key[
            (pair_id, "N")
        ] = "train"

    # N outside training; H counterpart in training.
    for pair_id in n_validation_pairs:
        split_by_key[
            (pair_id, "N")
        ] = "validation"
        split_by_key[
            (pair_id, "H")
        ] = "train"

    for pair_id in n_test_pairs:
        split_by_key[
            (pair_id, "N")
        ] = "test"
        split_by_key[
            (pair_id, "H")
        ] = "train"

    if len(split_by_key) != len(dataframe):
        raise RuntimeError(
            "Max-cross assignment did not cover all examples."
        )

    result = dataframe.copy()

    result["review_split"] = result.apply(
        lambda row: split_by_key[
            (
                row["pair_id"],
                row["pair_suffix"],
            )
        ],
        axis=1,
    )

    return result


## Generate and save all review splits

Each strategy/seed directory contains:

- `train.jsonl`
- `validation.jsonl`
- `test.jsonl`
- `metadata.json`
- `pair_matrix.csv`
- `inspection.csv`

Each strategy directory also receives a `summary.csv`.


In [20]:
generation_functions = {
    "pair_controlled": generate_pair_controlled,
    "random_instance": generate_random_instance,
    "max_cross_split": generate_max_cross_split,
}

all_summaries = []

for strategy in STRATEGIES:
    strategy_summaries = []

    print("=" * 80)
    print("Generating strategy:", strategy)

    for seed in SPLIT_SEEDS:
        generated = generation_functions[strategy](
            dataframe=df,
            seed=seed,
        )

        validation_summary = validate_generated_split(
            dataframe=generated,
            strategy=strategy,
            seed=seed,
        )

        output_dir = save_generated_run(
            dataframe=generated,
            strategy=strategy,
            seed=seed,
            validation_summary=validation_summary,
        )

        summary_row = {
            key: value
            for key, value in validation_summary.items()
            if key != "pair_matrix"
        }

        strategy_summaries.append(
            summary_row
        )
        all_summaries.append(
            summary_row
        )

        print(
            f"{strategy:18s} | seed={seed:2d} | "
            f"cross-split={summary_row['cross_split_pairs']:4d} "
            f"({summary_row['cross_split_rate']:.4%}) | "
            f"{output_dir}"
        )

    strategy_summary_df = pd.DataFrame(
        strategy_summaries
    )

    strategy_summary_path = (
        REVIEW_SPLITS_DIR
        / strategy
        / "summary.csv"
    )

    strategy_summary_df.to_csv(
        strategy_summary_path,
        index=False,
        encoding="utf-8",
    )

    print("Strategy summary saved to:", strategy_summary_path)


Generating strategy: pair_controlled
pair_controlled    | seed=13 | cross-split=   0 (0.0000%) | /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/seed_13
pair_controlled    | seed=21 | cross-split=   0 (0.0000%) | /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/seed_21
pair_controlled    | seed=40 | cross-split=   0 (0.0000%) | /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/seed_40
pair_controlled    | seed=42 | cross-split=   0 (0.0000%) | /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/seed_42
pair_controlled    | seed=73 | cross-split=   0 (0.0000%) | /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/seed_73
Strategy summary saved to: /home/avelar/pun-detection-split-analysis/data/review_splits/pair_controlled/summary.csv
Generating strategy: random_instance
random_instance    | seed=13 | cross-split=1329 (46.6316%) | /home/avelar/pun-detecti

In [21]:
all_summary_df = pd.DataFrame(
    all_summaries
)

all_summary_path = (
    REVIEW_SPLITS_DIR
    / "all_splits_summary.csv"
)

all_summary_df.to_csv(
    all_summary_path,
    index=False,
    encoding="utf-8",
)

print("All-splits summary:")
display(all_summary_df)

print("Saved to:", all_summary_path)


All-splits summary:


,strategy,seed,total_examples,total_pairs,train_examples,validation_examples,test_examples,cross_split_pairs,cross_split_rate
0,pair_controlled,13,5700,2850,3990,570,1140,0,0.000000
1,pair_controlled,21,5700,2850,3990,570,1140,0,0.000000
2,pair_controlled,40,5700,2850,3990,570,1140,0,0.000000
3,pair_controlled,42,5700,2850,3990,570,1140,0,0.000000
4,pair_controlled,73,5700,2850,3990,570,1140,0,0.000000
5,random_instance,13,5700,2850,3990,570,1140,1329,0.466316
6,random_instance,21,5700,2850,3990,570,1140,1291,0.452982
7,random_instance,40,5700,2850,3990,570,1140,1303,0.457193
8,random_instance,42,5700,2850,3990,570,1140,1326,0.465263
9,random_instance,73,5700,2850,3990,570,1140,1316,0.461754


Saved to: /home/avelar/pun-detection-split-analysis/data/review_splits/all_splits_summary.csv


In [22]:
# Add the original corpus as a structural reference in a comparison table.
original_reference = pd.DataFrame(
    [
        {
            "strategy": "original",
            "seed": np.nan,
            "total_examples": len(df),
            "total_pairs": df["pair_id"].nunique(),
            "train_examples": EXPECTED_SPLIT_COUNTS["train"],
            "validation_examples": EXPECTED_SPLIT_COUNTS["validation"],
            "test_examples": EXPECTED_SPLIT_COUNTS["test"],
            "cross_split_pairs": original_cross_split_pairs,
            "cross_split_rate": original_cross_split_rate,
        }
    ]
)

comparison_df = pd.concat(
    [
        original_reference,
        all_summary_df,
    ],
    ignore_index=True,
)

comparison_path = (
    REVIEW_SPLITS_DIR
    / "comparison_with_original.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8",
)

display(comparison_df)

print(
    "Comparison with original saved to:",
    comparison_path,
)


,strategy,seed,total_examples,total_pairs,train_examples,validation_examples,test_examples,cross_split_pairs,cross_split_rate
0,original,NaN,5700,2850,3990,570,1140,1306,0.458246
1,pair_controlled,13.0,5700,2850,3990,570,1140,0,0.000000
2,pair_controlled,21.0,5700,2850,3990,570,1140,0,0.000000
3,pair_controlled,40.0,5700,2850,3990,570,1140,0,0.000000
4,pair_controlled,42.0,5700,2850,3990,570,1140,0,0.000000
5,pair_controlled,73.0,5700,2850,3990,570,1140,0,0.000000
6,random_instance,13.0,5700,2850,3990,570,1140,1329,0.466316
7,random_instance,21.0,5700,2850,3990,570,1140,1291,0.452982
8,random_instance,40.0,5700,2850,3990,570,1140,1303,0.457193
9,random_instance,42.0,5700,2850,3990,570,1140,1326,0.465263


Comparison with original saved to: /home/avelar/pun-detection-split-analysis/data/review_splits/comparison_with_original.csv


## Final on-disk validation

This cell reopens every generated JSONL file and checks that the saved files have the expected number of lines.


In [23]:
saved_file_checks = []

for strategy in STRATEGIES:
    for seed in SPLIT_SEEDS:
        seed_dir = (
            REVIEW_SPLITS_DIR
            / strategy
            / f"seed_{seed}"
        )

        row = {
            "strategy": strategy,
            "seed": seed,
        }

        for split_name in SPLIT_NAMES:
            path = (
                seed_dir
                / f"{split_name}.jsonl"
            )

            if not path.is_file():
                raise FileNotFoundError(
                    f"Missing generated file: {path}"
                )

            observed_lines = count_jsonl_lines(
                path
            )

            expected_lines = (
                EXPECTED_SPLIT_COUNTS[
                    split_name
                ]
            )

            if observed_lines != expected_lines:
                raise ValueError(
                    f"{strategy}/{seed}/{split_name}: "
                    f"expected {expected_lines} lines, "
                    f"found {observed_lines}."
                )

            row[
                f"{split_name}_lines"
            ] = observed_lines

        saved_file_checks.append(row)

saved_file_checks_df = pd.DataFrame(
    saved_file_checks
)

display(saved_file_checks_df)

print("All generated JSONL files passed final on-disk validation.")


,strategy,seed,train_lines,validation_lines,test_lines
0,pair_controlled,13,3990,570,1140
1,pair_controlled,21,3990,570,1140
2,pair_controlled,40,3990,570,1140
3,pair_controlled,42,3990,570,1140
4,pair_controlled,73,3990,570,1140
5,random_instance,13,3990,570,1140
6,random_instance,21,3990,570,1140
7,random_instance,40,3990,570,1140
8,random_instance,42,3990,570,1140
9,random_instance,73,3990,570,1140


All generated JSONL files passed final on-disk validation.


In [24]:
print("\nFinal structural summary")
print("-" * 80)
print(
    f"Original:           {original_cross_split_pairs:4d}/2850 "
    f"({original_cross_split_rate:.4%}) cross-split pairs"
)

for strategy in STRATEGIES:
    strategy_rows = all_summary_df[
        all_summary_df["strategy"] == strategy
    ]

    print(
        f"{strategy:18s}: "
        f"cross-split pairs = "
        f"{strategy_rows['cross_split_pairs'].tolist()}"
    )

print("\nExpected invariants:")
print("- pair_controlled: 0 cross-split pairs for every seed")
print("- max_cross_split: 1710 cross-split pairs (60%) for every seed")
print("- random_instance: varies by seed")
print("- all generated splits: exact 3990/570/1140 sizes")
print("- all generated splits: exact 50/50 class balance")



Final structural summary
--------------------------------------------------------------------------------
Original:           1306/2850 (45.8246%) cross-split pairs
pair_controlled   : cross-split pairs = [0, 0, 0, 0, 0]
random_instance   : cross-split pairs = [1329, 1291, 1303, 1326, 1316]
max_cross_split   : cross-split pairs = [1710, 1710, 1710, 1710, 1710]

Expected invariants:
- pair_controlled: 0 cross-split pairs for every seed
- max_cross_split: 1710 cross-split pairs (60%) for every seed
- random_instance: varies by seed
- all generated splits: exact 3990/570/1140 sizes
- all generated splits: exact 50/50 class balance
